# MAKE A COPY OF THIS AND GIVE A UNIQUE NAME BEFORE EDITING

# Building and Automating an ETL Notebook in Databricks

In this notebook we will build a ETL pipeline — loading raw weather data, transforming it, and persisting curated results as a Delta table. Once built, this notebook can be scheduled as a Databricks Job to run automatically on a cadence.

**Components:**
1. Load the data
2. Transform the data
3. Persist the data
4. Review table transaction logs

   
## Quick Tour of the Databricks Notebook UI

Before diving into the code, here's a quick orientation of the key areas of the notebook interface:

- **Top Bar** — notebook title, compute selector, and the Run All / Schedule buttons
- **Sidebar (left)** — workspace file browser, search, and catalog explorer
- **Cell Toolbar** — each cell has controls to run, move, delete, or change language (Python, SQL, Scala, R, Markdown)
- **Widgets Bar** — parameters defined with `dbutils.widgets` appear here for interactive input
- **Results Panel** — execution output renders below each cell (tables, charts, text, errors)
- **Right Sidebar** — Databricks Assistant / Genie Code, comments, revision history, and environment panel (libraries, Spark config)
- **Status Bar (bottom)** — cluster state, Spark UI link, and cell execution progress

> **Tip:** You can mix languages in a single notebook by adding a magic command (`%sql`, `%python`, `%sh`, `%md`) at the top of any cell.

## 1. Notebook Setup - Overview
- installing dependencies
- using secrets
- defining parameters with widgets

In [0]:
# DO NOT MODIFY - FOR DEMO ONLY
current_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
unique_table_suffix = current_user.split('@')[0].replace('.', '_')

**Notebook dependencies can be installed in a cell or in the notebook environment panel**

A common practice is to keep project or team level dependencies in a `requirements.txt` file and store in a volume, this file can be users across many notebooks and ensure consistent package use

In [0]:
%sh
# Databricks compute comes pre-loaded with many libraries - this is just a example of how to install missing libraries
pip install --quiet pandas
pip install --quiet numpy

**You may need to access secrets, such as API keys or CLIENT-ID**

Databricks uses secret scopes to store secret values. Secret scopes can also point to a existing Azure KeyVault. 

In [0]:
secret_scope_name = "bu_your_business_unit"
secret_key = "my_secret_key"

# example only - if the scope and key existed you can retrieve with the following
# my_secret_value = dbutils.secrets.get(scope=secret_scope_name, key=secret_key)

**We can add notebook parameters that can be set through widgets and job parameters**

You may want to use a notebook across Dev/QA/Prod environments and point to different catalog and schema. Or be able to alter the value of a variable dynamically in a job or between tasks. Parameters help us do this. 

In [0]:
# create notebook widgets so we can pass dynamic parameters in jobs
dbutils.widgets.text("catalog", "classic_stable_been2c_catalog", "Catalog")
dbutils.widgets.text("schema", "weather", "Schema")

**If we change the widget value and run the below cell multiple times the output will also change**

In [0]:
# get widgets values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print("catalog: ", catalog)
print("schema: ", schema)

## 2. Load in Data

There are several ways to bring data into a notebook. We'll cover three common patterns:
- **SQL temp view** — query Unity Catalog tables using SQL
- **PySpark DataFrame** — read a table directly into a Spark Dataframe
- **File-based** — read CSV or Excel from a Databricks Volume

**Option 1: Create a Temp View and Query with SQL**

Temp views exist only for the duration of the session. They're useful when you want to break SQL work into reusable chunks without writing anything to the catalog.

We are also using our dynamic catalog and schema - it will become apparent why when we create a job.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_view AS
SELECT * FROM classic_stable_been2c_catalog.weather.us_postal_daily_metric;

SELECT * FROM temp_view LIMIT 5;

**Option 2a: Read a Unity Catalog Table into a PySpark DataFrame**

Reading directly into a DataFrame gives you the full power of PySpark for transformations. `spark.table()` takes a three-level Unity Catalog name: `catalog.schema.table`.

In [0]:
df = spark.table(f'{catalog}.{schema}.us_postal_daily_metric')

display(df.limit(5))

   
**Option 2b: Use Python to Pass Parameters into SQL**

You can also run SQL from Python using f-strings to inject widget values and variables directly into the query. This gives you the flexibility of SQL with the control of Python — useful when filters or table references need to be set dynamically.

In [0]:
## Example using python to pass parameters to SQL
station_code_filter = 'NQI'

df = sql(

    f"""
    
    SELECT * FROM {catalog}.{schema}.us_postal_daily_metric
    WHERE station_code = '{station_code_filter}'
    
    """
)

display(df.limit(5))

%md
**Option 2c: Convert the PySpark Dataframe into Pandas Dataframe**

Generally we recommend keeping dataframes in PySpark, but sometimes you may be more comfortable working in Pandas. This is suitable for small datasets.  

In [0]:
df_pandas = df.toPandas()

display(df_pandas.head(5))

**Option 3a: Read a CSV from a Databricks Volume**

Databricks Volumes provide governed storage for files. You can upload CSV, Excel, JSON, and other files to a Volume and read them directly in a notebook using the `/Volumes/` path prefix.

In [0]:
df = spark.read.csv(f"/Volumes/{catalog}/{schema}/files/weather.csv", header=True, inferSchema=True)

df.limit(5).display()

**Option 3b: Read an Excel file from a Databricks Volume**

Excel files can also be read from Volumes. Databricks handles the parsing — no extra library installation needed.

In [0]:
df = spark.read.format("excel").option("header", True).load(f"/Volumes/{catalog}/{schema}/files/weather.xlsx")

display(df.limit(5))

## 3. Transform the Data

Using PySpark we'll clean and aggregate the raw data into a curated dataset. We'll work through three common transformation steps: selecting relevant columns, removing bad data, and aggregating to a useful grain.

**Start with a fresh copy of the source data**

It's good practice to reload from the source at the start of your transformation chain so the notebook is idempotent — it produces the same result every time it runs.

In [0]:
df = spark.table(f"{catalog}.{schema}.us_postal_daily_metric")

display(df.limit(5))

**Step 1: Select only the columns you need**

Dropping unused columns early reduces memory usage and keeps downstream logic easier to follow.

In [0]:
from pyspark.sql.types import DoubleType

df = df.select(
    "latitude", "longitude", "station_code", "date", "phrase_long", "temperature_avg", "wind_speed_avg", "rain_lwe_total"
)

display(df.limit(5))

**Step 2: Remove bad data**

Drop rows with nulls and filter out known bad records. In a production pipeline you'd typically log the dropped rows rather than silently discard them.

In [0]:
df = df.dropna().filter(df.station_code != "NYCG")

display(df.limit(5))

**Step 3: Aggregate to monthly averages**

Roll up the daily records to monthly averages per station. This is the curated grain we'll persist as a Delta table.

In [0]:
from pyspark.sql.functions import month, avg

df_monthly_avg = df.withColumn("month", month("date")) \
    .groupBy("station_code", "month") \
    .agg(
        avg("temperature_avg").alias("avg_temperature"),
        avg("wind_speed_avg").alias("avg_wind_speed"),
        avg("rain_lwe_total").alias("avg_rain_lwe")
    )

display(df_monthly_avg.limit(5))

## 3. Persist the Data

After transforming the data, we need to write it somewhere durable so other users and tools can access it. There are three main patterns:

| Pattern | When to use |
|---|---|
| **View** | Ad-hoc / lightweight — no data stored, transformation re-runs on every query |
| **Overwrite table** | Full refresh — simplest option, replaces the entire table each job run |
| **Delta merge (upsert)** | Incremental updates — adds new rows and updates changed ones, most production-ready |

**Option 1: Create a View**

A view stores the SQL definition, not the data. Every query re-runs the transformation against the source table — useful for always-fresh, ad-hoc datasets. For scheduled jobs that serve downstream users, a persisted table is usually a better choice.

In [0]:
sql(f"""
CREATE OR REPLACE VIEW {catalog}.{schema}.monthly_avg_view_{unique_table_suffix} AS
SELECT
    station_code,
    MONTH(date) AS month,
    AVG(temperature_avg) AS avg_temperature,
    AVG(wind_speed_avg) AS avg_wind_speed,
    AVG(precipitation_probability) AS avg_precip_probability,
    AVG(rain_lwe_total) AS avg_rain_lwe
FROM {catalog}.{schema}.us_postal_daily_metric
GROUP BY station_code, MONTH(date)
""")

# read back in to show
display(sql(f"SELECT * FROM {catalog}.{schema}.monthly_avg_view_{unique_table_suffix} LIMIT 5"))

**Option 2: Overwrite — Write a Full Table on Every Run**

The simplest persistence pattern: replace the entire table each time the job runs. Works well when the source dataset is small enough to fully reprocess and you don't need to preserve history.

In [0]:
df_monthly_avg.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.monthly_avg_{unique_table_suffix}")

# read back in to show
df = spark.table(f"{catalog}.{schema}.monthly_avg_{unique_table_suffix}")

display(df.limit(5))

**Option 3: Delta Merge — Upsert New and Changed Rows**

When new data arrives incrementally, a merge (upsert) is more efficient than a full overwrite. Delta's merge operation matches records on a key, updates rows that already exist, and inserts rows that are new.

In [0]:
from pyspark.sql import Row
from delta.tables import DeltaTable

# Simulating incoming data - first row exists and we are updating, next row is new
new_data = [
    Row(station_code="ORNY", month=4, avg_temperature=14.174545, avg_wind_speed=2.803636, avg_precip_probability=17.90909090909091, avg_rain_lwe=0.078509091),
    Row(station_code="ABC",  month=4, avg_temperature=15.132727, avg_wind_speed=2.845455, avg_precip_probability=41.81818181818182, avg_rain_lwe=2.496131818)
]

df_new = spark.createDataFrame(new_data)

display(df_new)

In [0]:
# Use merge to update and insert new records
delta_table = DeltaTable.forName(spark, f"{catalog}.{schema}.monthly_avg_{unique_table_suffix}")

delta_table.alias("target").merge(
    df_new.alias("source"),
    "target.station_code = source.station_code AND target.month = source.month"
).whenMatchedUpdate(set={
    "avg_temperature": "source.avg_temperature",
    "avg_wind_speed": "source.avg_wind_speed",
    "avg_rain_lwe": "source.avg_rain_lwe"
}).whenNotMatchedInsert(values={
    "station_code": "source.station_code",
    "month": "source.month",
    "avg_temperature": "source.avg_temperature",
    "avg_wind_speed": "source.avg_wind_speed",
    "avg_rain_lwe": "source.avg_rain_lwe"
}).execute()

# read back in to show
display(spark.table(f"{catalog}.{schema}.monthly_avg_{unique_table_suffix}").limit(5))

## 4. Delta Transaction Logs

Every write to a Delta table is recorded in a transaction log. This gives you a full audit trail of every change — who wrote what, when, and using which operation (overwrite, merge, etc.).

You can also use this log to **time-travel**: query the table as it looked at any previous version.

In [0]:
delta_table = DeltaTable.forName(spark, f"{catalog}.{schema}.monthly_avg_{unique_table_suffix}")

display(delta_table.history().limit(5))

**Time Travel — Query a Previous Version**

You can read any previous version of the table using `VERSION AS OF` in SQL, or the `.option("versionAsOf", ...)` reader option in PySpark.

In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table(f"{catalog}.{schema}.monthly_avg_{unique_table_suffix}")

display(df_v0.limit(5))

**Grant Use of New Table to a Group**

You can grant a group `SELECT` or more permissions using SQL or through the Unity Catalog UI

In [0]:
# Grant access
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `my_group`")

spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `my_group`")

spark.sql(f"GRANT SELECT ON TABLE {catalog}.{schema}.monthly_avg_{unique_table_suffix} TO `my_group`")

# View current grants
display(spark.sql(f"SHOW GRANTS ON TABLE {catalog}.{schema}.monthly_avg_{unique_table_suffix}"))